# m_010 – Shallow Random Forest with Logistic Regression at Leaves

In [1]:
# Content: define metadata, load data, checks, model, train, and score.
MODEL_ID = '010'
RUNNER = 'codex'
MACHINE = 'cpu'
DATA_SCOPE = 'full'

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import importlib.util

X_train = pd.read_csv('../training_data/train_df_factor.csv')
y_train = pd.read_csv('../training_data/train_df_target.csv')['Prepaied_3m'].astype(int)
X_val = pd.read_csv('../val_data/val_df_factor.csv')
y_val = pd.read_csv('../val_data/val_df_target.csv')['Prepaied_3m'].astype(int)

print('train factor shape:', X_train.shape)
print('train target shape:', y_train.shape)
print('val factor shape:', X_val.shape)
print('val target shape:', y_val.shape)
print('row alignment(train):', len(X_train) == len(y_train))
print('row alignment(val):', len(X_val) == len(y_val))
print('train positive rate:', float(y_train.mean()))
print('val positive rate:', float(y_val.mean()))

class TreeWithLogisticLeaves(BaseEstimator, ClassifierMixin):
    def __init__(self, max_depth=3, min_samples_leaf=120, C=0.5):
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.C = C

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.tree_ = DecisionTreeClassifier(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf, class_weight='balanced', random_state=42)
        self.tree_.fit(X, y)
        leaves = self.tree_.apply(X)
        self.models_ = {}
        self.default_prob_ = float(np.clip(y.mean(), 1e-6, 1 - 1e-6))
        self.classes_ = np.array([0, 1])
        for leaf_id in np.unique(leaves):
            idx = leaves == leaf_id
            X_leaf, y_leaf = X[idx], y[idx]
            if len(np.unique(y_leaf)) < 2 or len(y_leaf) < 30:
                self.models_[leaf_id] = ('constant', float(np.clip(y_leaf.mean(), 1e-6, 1 - 1e-6)))
            else:
                lr = LogisticRegression(C=self.C, max_iter=3000, class_weight='balanced', solver='liblinear')
                lr.fit(X_leaf, y_leaf)
                self.models_[leaf_id] = ('lr', lr)
        return self

    def predict_proba(self, X):
        X = np.asarray(X)
        leaves = self.tree_.apply(X)
        probs = np.zeros((X.shape[0], 2), dtype=float)
        for i, leaf_id in enumerate(leaves):
            model_type, model_obj = self.models_.get(leaf_id, ('constant', self.default_prob_))
            p1 = model_obj.predict_proba(X[i:i+1])[0, 1] if model_type == 'lr' else model_obj
            probs[i, 1] = float(np.clip(p1, 1e-6, 1 - 1e-6))
            probs[i, 0] = 1.0 - probs[i, 1]
        return probs

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.6).astype(int)

numeric_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
])

forest_tree_lr = BaggingClassifier(
    estimator=TreeWithLogisticLeaves(max_depth=3, min_samples_leaf=120, C=0.5),
    n_estimators=20,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42,
    n_jobs=1,
)

pipeline = Pipeline([('prep', preprocess), ('clf', forest_tree_lr)])

param_grid = {
    'clf__n_estimators': [20, 40],
    'clf__max_samples': [0.8],
    'clf__max_features': [0.8],
    'clf__estimator__max_depth': [3],
    'clf__estimator__min_samples_leaf': [120],
    'clf__estimator__C': [0.5],
}

cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
search = GridSearchCV(pipeline, param_grid=param_grid, scoring='balanced_accuracy', cv=cv, n_jobs=1, verbose=1)
search.fit(X_train, y_train)

print('best balanced_accuracy (cv):', search.best_score_)
print('best params:', search.best_params_)

best_model = search.best_estimator_
val_pred = best_model.predict(X_val)

spec = importlib.util.spec_from_file_location('validation_score', '../help_stuff/validation_score.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
score = mod.prediction_score(val_pred)
print('validation score:', score)






/tmp/ipykernel_20950/181244207.py:20: DtypeWarning: Columns (0: Repurchase Make Whole Proceeds Flag) have mixed types. Specify dtype option on import or set low_memory=False.
  X_train = pd.read_csv('../training_data/train_df_factor.csv')


train factor shape: (45062, 74)
train target shape: (45062,)
val factor shape: (104, 74)
val target shape: (104,)
row alignment(train): True
row alignment(val): True
train positive rate: 0.004127646353912388
val positive rate: 0.5
Fitting 2 folds for each of 2 candidates, totalling 4 fits


/root/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['UPB at Issuance'
 'Interest Only First Principal And Interest Payment Date'
 'Months to Amortization' 'Mortgage Insurance Cancellation Indicator'
 'Scheduled Principal Current' 'Property Preservation and Repair Costs'
 'Asset Recovery Costs' 'Miscellaneous Holding Expenses and Credits'
 'Associated Taxes for Holding Property'
 'Modification-Related Non-Interest Bearing UPB'
 'Principal Forgiveness Amount' 'Original List Start Date'
 'Original List Price' 'Current List Price'
 'Borrower Credit Score At Issuance'
 'Co-Borrower Credit Score At Issuance' 'Borrower Credit Score Current '
 'Co-Borrower Credit Score Current' 'Loan Holdback Effective Date'
 'ARM Initial Fixed-Rate Period  ≤ 5 YR Indicator' 'ARM Product Type'
 'Initial Fixed-Rate Period ' 'Interest Rate Adjustment Frequency'
 'Next Interest Rate Adjustment Date' 'Next Payment Chan

/root/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['UPB at Issuance'
 'Interest Only First Principal And Interest Payment Date'
 'Months to Amortization' 'Mortgage Insurance Cancellation Indicator'
 'Scheduled Principal Current' 'Property Preservation and Repair Costs'
 'Asset Recovery Costs' 'Miscellaneous Holding Expenses and Credits'
 'Associated Taxes for Holding Property'
 'Modification-Related Non-Interest Bearing UPB'
 'Principal Forgiveness Amount' 'Original List Start Date'
 'Original List Price' 'Current List Price'
 'Borrower Credit Score At Issuance'
 'Co-Borrower Credit Score At Issuance' 'Borrower Credit Score Current '
 'Co-Borrower Credit Score Current' 'Loan Holdback Effective Date'
 'ARM Initial Fixed-Rate Period  ≤ 5 YR Indicator' 'ARM Product Type'
 'Initial Fixed-Rate Period ' 'Interest Rate Adjustment Frequency'
 'Next Interest Rate Adjustment Date' 'Next Payment Chan

/root/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['UPB at Issuance'
 'Interest Only First Principal And Interest Payment Date'
 'Months to Amortization' 'Mortgage Insurance Cancellation Indicator'
 'Scheduled Principal Current' 'Property Preservation and Repair Costs'
 'Asset Recovery Costs' 'Miscellaneous Holding Expenses and Credits'
 'Associated Taxes for Holding Property'
 'Modification-Related Non-Interest Bearing UPB'
 'Principal Forgiveness Amount' 'Original List Start Date'
 'Original List Price' 'Current List Price'
 'Borrower Credit Score At Issuance'
 'Co-Borrower Credit Score At Issuance' 'Borrower Credit Score Current '
 'Co-Borrower Credit Score Current' 'Loan Holdback Effective Date'
 'ARM Initial Fixed-Rate Period  ≤ 5 YR Indicator' 'ARM Product Type'
 'Initial Fixed-Rate Period ' 'Interest Rate Adjustment Frequency'
 'Next Interest Rate Adjustment Date' 'Next Payment Chan

...

/root/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['UPB at Issuance'
 'Interest Only First Principal And Interest Payment Date'
 'Months to Amortization' 'Mortgage Insurance Cancellation Indicator'
 'Scheduled Principal Current' 'Property Preservation and Repair Costs'
 'Asset Recovery Costs' 'Miscellaneous Holding Expenses and Credits'
 'Associated Taxes for Holding Property'
 'Modification-Related Non-Interest Bearing UPB'
 'Principal Forgiveness Amount' 'Original List Start Date'
 'Original List Price' 'Current List Price'
 'Borrower Credit Score At Issuance'
 'Co-Borrower Credit Score At Issuance' 'Borrower Credit Score Current '
 'Co-Borrower Credit Score Current' 'Loan Holdback Effective Date'
 'ARM Initial Fixed-Rate Period  ≤ 5 YR Indicator' 'ARM Product Type'
 'Initial Fixed-Rate Period ' 'Interest Rate Adjustment Frequency'
 'Next Interest Rate Adjustment Date' 'Next Payment Chan

/root/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['UPB at Issuance'
 'Interest Only First Principal And Interest Payment Date'
 'Months to Amortization' 'Mortgage Insurance Cancellation Indicator'
 'Scheduled Principal Current' 'Property Preservation and Repair Costs'
 'Asset Recovery Costs' 'Miscellaneous Holding Expenses and Credits'
 'Associated Taxes for Holding Property'
 'Modification-Related Non-Interest Bearing UPB'
 'Principal Forgiveness Amount' 'Original List Start Date'
 'Original List Price' 'Current List Price'
 'Borrower Credit Score At Issuance'
 'Co-Borrower Credit Score At Issuance' 'Borrower Credit Score Current '
 'Co-Borrower Credit Score Current' 'Loan Holdback Effective Date'
 'ARM Initial Fixed-Rate Period  ≤ 5 YR Indicator' 'ARM Product Type'
 'Initial Fixed-Rate Period ' 'Interest Rate Adjustment Frequency'
 'Next Interest Rate Adjustment Date' 'Next Payment Chan

best balanced_accuracy (cv): 0.5313727096985049
best params: {'clf__estimator__C': 0.5, 'clf__estimator__max_depth': 3, 'clf__estimator__min_samples_leaf': 120, 'clf__max_features': 0.8, 'clf__max_samples': 0.8, 'clf__n_estimators': 20}


/root/.pyenv/versions/3.12.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['UPB at Issuance'
 'Interest Only First Principal And Interest Payment Date'
 'Months to Amortization' 'Mortgage Insurance Cancellation Indicator'
 'Scheduled Principal Current' 'Property Preservation and Repair Costs'
 'Asset Recovery Costs' 'Miscellaneous Holding Expenses and Credits'
 'Associated Taxes for Holding Property'
 'Modification-Related Non-Interest Bearing UPB'
 'Principal Forgiveness Amount' 'Original List Start Date'
 'Original List Price' 'Current List Price'
 'Borrower Credit Score At Issuance'
 'Co-Borrower Credit Score At Issuance' 'Borrower Credit Score Current '
 'Co-Borrower Credit Score Current' 'Loan Holdback Effective Date'
 'ARM Initial Fixed-Rate Period  ≤ 5 YR Indicator' 'ARM Product Type'
 'Initial Fixed-Rate Period ' 'Interest Rate Adjustment Frequency'
 'Next Interest Rate Adjustment Date' 'Next Payment Chan

validation score: 0.5192307692307692
